In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    accuracy_score
)
from sklearn.ensemble import RandomForestClassifier

In [ ]:


# ----------------------------------
# 1. Load processed dataset
# ----------------------------------
DATA_PATH = "../data/processed/"
df = pd.read_parquet(f"{DATA_PATH}cleaned_feature_selected.parquet")

print(f"Dataset shape: {df.shape}")
print(f"Attack ratio: {df['is_attack'].mean():.4f}")

# ----------------------------------
# 2. Define targets and features
# ----------------------------------
TARGET_COLS = [
    'is_attack',
    'attack_category',
    'label',
    'difficulty_level',
    'anomaly_score',
    'potential_zero_day'
]

X = df.drop(columns=TARGET_COLS, errors="ignore")

y_binary = df['is_attack']
y_multi = df['attack_category']

# Encode multiclass labels
le_multi = LabelEncoder()
y_multi_enc = le_multi.fit_transform(y_multi)

# ----------------------------------
# 3. Identify categorical & numerical features
# ----------------------------------
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

print(f"Numerical features: {len(numerical_cols)}")
print(f"Categorical features: {len(categorical_cols)}")

# ----------------------------------
# 4. Preprocessing (ONLY encode categoricals)
# ----------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', LabelEncoder(), categorical_cols)
    ],
    remainder='passthrough'
)

# ⚠️ LabelEncoder cannot be used directly in ColumnTransformer
# → So we encode categoricals manually

X_encoded = X.copy()
for col in categorical_cols:
    X_encoded[col] = LabelEncoder().fit_transform(X_encoded[col])

# ----------------------------------
# 5. Train-test split
# ----------------------------------
X_train, X_test, y_bin_train, y_bin_test = train_test_split(
    X_encoded,
    y_binary,
    test_size=0.2,
    stratify=y_binary,
    random_state=42
)

_, _, y_multi_train, y_multi_test = train_test_split(
    X_encoded,
    y_multi_enc,
    test_size=0.2,
    stratify=y_multi_enc,
    random_state=42
)

# ----------------------------------
# 6. Baseline Model (Random Forest)
# ----------------------------------
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\n" + "="*60)
print("BASELINE MODEL: RANDOM FOREST")
print("="*60)

# ==================================================
# 7. Binary Classification
# ==================================================
print("\n[1] Binary Classification (is_attack)")

rf.fit(X_train, y_bin_train)

y_bin_pred = rf.predict(X_test)
y_bin_prob = rf.predict_proba(X_test)[:, 1]

bin_f1 = f1_score(y_bin_test, y_bin_pred)
bin_auc = roc_auc_score(y_bin_test, y_bin_prob)

tn, fp, fn, tp = confusion_matrix(y_bin_test, y_bin_pred).ravel()
detection_rate = tp / (tp + fn)
false_alarm_rate = fp / (fp + tn)

print(f"F1 Score:        {bin_f1:.4f}")
print(f"ROC AUC:         {bin_auc:.4f}")
print(f"Detection Rate:  {detection_rate:.4f}")
print(f"False Alarm Rate:{false_alarm_rate:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_bin_test,
    y_bin_pred,
    target_names=["Normal", "Attack"]
))

# ==================================================
# 8. Multiclass Classification
# ==================================================
print("\n[2] Multiclass Classification (attack_category)")

rf.fit(X_train, y_multi_train)

y_multi_pred = rf.predict(X_test)

multi_f1_macro = f1_score(y_multi_test, y_multi_pred, average="macro")
multi_acc = accuracy_score(y_multi_test, y_multi_pred)

print(f"Macro F1 Score: {multi_f1_macro:.4f}")
print(f"Accuracy:       {multi_acc:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_multi_test,
    y_multi_pred,
    target_names=le_multi.classes_
))

# ==================================================
# 9. Cross-validation (stability check)
# ==================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_bin = cross_val_score(
    rf, X_encoded, y_binary, cv=cv, scoring="f1"
)

cv_multi = cross_val_score(
    rf, X_encoded, y_multi_enc, cv=cv, scoring="f1_macro"
)

print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS")
print("="*60)
print(f"Binary F1:            {cv_bin.mean():.4f} ± {cv_bin.std():.4f}")
print(f"Multiclass Macro F1:  {cv_multi.mean():.4f} ± {cv_multi.std():.4f}")

# ==================================================
# 10. Save baseline predictions (optional)
# ==================================================
baseline_results = X_test.copy()
baseline_results["true_is_attack"] = y_bin_test.values
baseline_results["pred_is_attack"] = y_bin_pred
baseline_results["true_attack_category"] = y_multi_test
baseline_results["pred_attack_category"] = y_multi_pred

# baseline_results.to_csv("baseline_predictions.csv", index=False)

print("\nBaseline predictions saved to 'baseline_predictions.csv'")


Dataset shape: (125973, 47)
Attack ratio: 0.4654
Numerical features: 39
Categorical features: 4

BASELINE MODEL: RANDOM FOREST

[1] Binary Classification (is_attack)
F1 Score:        0.9988
ROC AUC:         1.0000
Detection Rate:  0.9983
False Alarm Rate:0.0005

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     13469
      Attack       1.00      1.00      1.00     11726

    accuracy                           1.00     25195
   macro avg       1.00      1.00      1.00     25195
weighted avg       1.00      1.00      1.00     25195


[2] Multiclass Classification (attack_category)
Macro F1 Score: 0.1868
Accuracy:       0.4728

Classification Report:
              precision    recall  f1-score   support

         dos       0.35      0.26      0.30      9186
      normal       0.53      0.71      0.61     13469
       probe       0.08      0.02      0.03      2331
         r2l       0.00      0.00      0.00       1